# סמן AI — ניסוי Binoculars (DictaLM 2.0)
**לפני ההרצה, פעם אחת:**
1. בתפריט: **Runtime → Change runtime type → T4 GPU** → Save.
2. בצד שמאל לוחצים על סמל **המפתח 🔑 (Secrets)** → **Add new secret**: שם `REQ_AGE_KEY`, ערך = המפתח (השורה שמתחילה ב-`AGE-SECRET-KEY-`). מסמנים **Notebook access**.
3. **Runtime → Run all**.

הקלט מוצפן ומפוענח רק כאן; התוצאה מוצפנת לפני שהיא יורדת. בסוף יורד קובץ `bino_scores.jsonl.age` — שלחו אותו לClaude בצ'אט.
רק טקסטים ציבוריים מהרשת וטקסטים שכתבו צ'טבוטים; בלי וואטסאפ ובלי טקסטים שהודבקו.

In [ ]:
!pip -q install pyrage bitsandbytes accelerate
import torch; print("GPU:", torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else "")
assert torch.cuda.is_available(), "Runtime -> Change runtime type -> T4 GPU" 

In [ ]:
import gzip, json, urllib.request
from google.colab import userdata
from pyrage import decrypt, encrypt, x25519
key = x25519.Identity.from_str(userdata.get("REQ_AGE_KEY").strip())
base = "https://raw.githubusercontent.com/God-of-Data/hebai-public-tools"
blob = urllib.request.urlopen(f"{base}/bino-input/bino/windows.jsonl.gz.age").read()
rows = [json.loads(l) for l in gzip.decompress(decrypt(blob, [key])).decode().splitlines() if l.strip()]
rcp = x25519.Recipient.from_str([w for w in urllib.request.urlopen(f"{base}/main/recipient.txt").read().decode().split() if w.startswith("age1")][0])
print("texts:", len(rows))

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
q = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_compute_dtype=torch.float16, bnb_4bit_quant_type="nf4")
tok = AutoTokenizer.from_pretrained("dicta-il/dictalm2.0")
if tok.pad_token is None: tok.pad_token = tok.eos_token
tok.padding_side = "right"
obs = AutoModelForCausalLM.from_pretrained("dicta-il/dictalm2.0", quantization_config=q, device_map="cuda:0").eval()
perf = AutoModelForCausalLM.from_pretrained("dicta-il/dictalm2.0-instruct", quantization_config=q, device_map="cuda:0").eval()
print("models loaded; GPU memory GB:", round(torch.cuda.memory_allocated() / 1e9, 1))

In [ ]:
# Binoculars (Hans et al. 2024): score = log-perplexity under the performer / cross-perplexity(observer, performer).
# Low score -> machine-like. Also saves each model's own log-perplexity.
import time, torch.nn.functional as F
out, t0, B = [], time.time(), 16
order = sorted(range(len(rows)), key=lambda i: len(rows[i]["text"]))
with torch.no_grad():
    for s in range(0, len(order), B):
        idx = order[s:s + B]
        enc = tok([rows[i]["text"] for i in idx], return_tensors="pt", padding=True, truncation=True, max_length=192).to("cuda:0")
        lo = obs(**enc).logits[:, :-1].float(); lp = perf(**enc).logits[:, :-1].float()
        tgt = enc.input_ids[:, 1:]; m = enc.attention_mask[:, 1:].float()
        nll_p = F.cross_entropy(lp.transpose(1, 2), tgt, reduction="none")
        nll_o = F.cross_entropy(lo.transpose(1, 2), tgt, reduction="none")
        xent = -(F.softmax(lo, -1) * F.log_softmax(lp, -1)).sum(-1)
        n = m.sum(1).clamp(min=1)
        ppl_p = (nll_p * m).sum(1) / n; ppl_o = (nll_o * m).sum(1) / n; x = (xent * m).sum(1) / n
        for j, i in enumerate(idx):
            out.append({"id": rows[i]["id"], "ntok": int(n[j]), "logppl_perf": round(float(ppl_p[j]), 5),
                        "logppl_obs": round(float(ppl_o[j]), 5), "xppl": round(float(x[j]), 5),
                        "bino": round(float(ppl_p[j] / x[j]), 5)})
        if (s // B) % 100 == 0: print(f"{len(out)}/{len(rows)}  {time.time() - t0:.0f}s", flush=True)
print("done", len(out), f"{time.time() - t0:.0f}s")

In [ ]:
from google.colab import files
open("bino_scores.jsonl.age", "wb").write(encrypt("\n".join(json.dumps(r) for r in out).encode(), [rcp]))
files.download("bino_scores.jsonl.age")